# 1. import library

In [ ]:
import os
import glob
from tqdm import tqdm

import numpy as np
import pandas as pd
from scipy.optimize import curve_fit

import matplotlib as mpl
import matplotlib.pyplot as plt

# 2. Load data

In [ ]:
def load_csv(genealogy_file_list):
    all_spot_data = []

    for file in genealogy_file_list:
        df = pd.read_csv(file)

        # add some col
        file_name = os.path.basename(file)
        df['file_name'] = file_name

        if 'PY1' in file:
            df['Specie'] = 'PY1'
            df['Time'] = (df['Frame'] *10/60*10).round(2)
        elif 'europaea' in file:
            df['Specie'] = 'europaea'
            df['Time'] = ( df['Frame'] *10/60*5).round(2)
        else:
            df['Specie'] = 'Unknown'

        # trasnlate pixel to micron
        scale = 0.0645 # 0.0645 µm/pixel
        df["Area"] = df["Area"] * scale * scale

        all_spot_data.append(df)

    All_spot_data = pd.concat(all_spot_data, ignore_index=True)
    
    All_spot_data_2 = All_spot_data.copy()
    All_spot_data_2['replicate'] = All_spot_data_2['file_name'].apply(
        lambda x: 1 if 'N1_' in x else 2 if 'N2_' in x else 3 if 'N3_' in x else 0
    )
    All_spot_data_2['FOV'] = All_spot_data_2['file_name'].apply(
        lambda x: 1 if '_1' in x else 2 if '_2' in x else 3 if '_3' in x else 0
    )
    All_spot_data_2['Condition'] = All_spot_data_2['file_name'].apply(
        lambda x: 'no-supernatant' if '_no-supernatant' in x
        else 'supernatant' if '_supernatant' in x
        else ''
    )

    cols = All_spot_data_2.columns.tolist()
    cols_reordered = ['Specie', 'Condition', 'replicate', 'FOV', 'Time'] + [col for col in cols if col not in ['Specie', 'Condition', 'replicate', 'FOV', 'Time', 'file_name']]
    All_spot_data_2 = All_spot_data_2[cols_reordered]
    
    return(All_spot_data_2)


In [ ]:
def mutate_allspot_data(All_spot_data):
    # Calculate the total cell area in each field of view (FOV) over time
    All_spot_data_mutate = (
        All_spot_data
        .groupby(['Specie', 'Condition', 'replicate', 'FOV', 'Time'], as_index=False)
        .agg(um2_biomass = ('Area', 'sum'),
             cell_num = ('Area', 'size'))
    )
    
    return(All_spot_data_mutate)

In [ ]:
def limit_until_max_biomass(df, specie, condition, replicate, fov):
    # Limit the analysis to a single FOV (N. europaea, replicate 3, FOV 2) because cells flowed out during the later observation phase.
    mask = (
        (df['Specie'] == specie) &
        (df['Condition'] == condition) &
        (df['replicate'] == replicate) &
        (df['FOV'] == fov)
    )
    
    target_df = df.loc[mask]
    if target_df.empty:
        print("no data")
        return df

    t_max = target_df.loc[
        target_df['um2_biomass'].idxmax(), 'Time'
    ]
    df_limited = df.copy()
    df_limited = df_limited.loc[
        ~mask | (df_limited['Time'] <= t_max)
    ]

    return df_limited

In [ ]:
allspot_file_list = glob.glob("../0_rawdata/all_spots_PY1/*.csv") + glob.glob("../0_rawdata/all_spots_europaea/*.csv")
All_spot_data_raw = load_csv(allspot_file_list)

All_spot_data_mutate = mutate_allspot_data(All_spot_data_raw)
All_spot_data_mutate_limited = limit_until_max_biomass(All_spot_data_mutate, 
                                                       "europaea", "no-supernatant", 3, 2)

# 3. fit biomass data to exponential model

In [ ]:
def biomass_exp_fit(biomass_data):
    def exp_growth(t, um2_biomass0, k):
        return um2_biomass0 * np.exp(k * t)

    fit_results = []
    for (specie, cond, rep, fov), group in tqdm(biomass_data.groupby(["Specie", "Condition", "replicate", "FOV"])):
        group = group.sort_values("Time")
        t = group["Time"].values
        um2_biomass = group["um2_biomass"].values
        
        try:
            popt, pcov = curve_fit(exp_growth, t, um2_biomass, p0=[um2_biomass[0], 0.01])
            um2_biomass0, k = popt
            residuals = um2_biomass - exp_growth(t, *popt)
            ss_res = np.sum(residuals**2)
            ss_tot = np.sum((um2_biomass - np.mean(um2_biomass))**2)
            r_squared = 1 - (ss_res / ss_tot)

            fit_results.append({
                "Specie": specie,
                "Condition": cond,
                "replicate": rep,
                "FOV": fov,
                "um2_biomass0": um2_biomass0,
                "k": k,
                "r_squared": r_squared,
                "n_points": len(t),
                "cell_num_t0": group.loc[group["Time"].idxmin(), "cell_num"],
                "max_time": np.max(group["Time"])
            })

        except RuntimeError:
            continue

    fit_df = pd.DataFrame(fit_results)
    
    return(fit_df)

In [ ]:
fit_df = biomass_exp_fit(All_spot_data_mutate_limited)

# 4. Plot

In [ ]:
def set_mytheme_paper(ax):
    # font
    plt.rcParams["text.usetex"] = False
    plt.rcParams["font.family"] = "Helvetica"
    plt.rcParams["font.size"] = 8
    plt.rcParams["text.color"] = "black"
    mpl.rcParams['svg.fonttype'] = 'none'

    # title
    ax.title.set_fontsize(9.5)
    ax.title.set_color("black")
    ax.title.set_position((0.5, 1.05))

    # axis title
    ax.xaxis.label.set_size(8)
    ax.yaxis.label.set_size(8)
    ax.xaxis.label.set_color("black")
    ax.yaxis.label.set_color("black")

    # ticks
    ax.tick_params(axis='x', labelsize=6.5, colors="black")
    ax.tick_params(axis='y', labelsize=6.5, colors="black")

    # spine
    for spine in ax.spines.values():
        spine.set_color("black")
        spine.set_linewidth(1.0)

    # figure bacground
    ax.set_facecolor("none")
    ax.figure.set_facecolor("none")

    # grid
    ax.grid(False)

    # legend
    ax.legend(
        loc="upper left",
        frameon=False,
        fontsize=6.5
    )
    

In [ ]:
def plot_fitting_result(fit_df, All_spot_data_mutate, output_folder, IF_show):
    def exp_growth(t, um2_biomass0, k):
        return um2_biomass0 * np.exp(k * t)
    
    n_cols = 3
    n_rows = 9
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(3.2*n_cols, 2.4*n_rows))
    axes = axes.flatten()

    for ax, (_, row) in zip(axes, fit_df.iterrows()):
        specie = row["Specie"]
        condition = row["Condition"]
        rep = row["replicate"]
        fov = row["FOV"]

        group = All_spot_data_mutate[
            (All_spot_data_mutate["Specie"] == specie) &
            (All_spot_data_mutate["Condition"] == condition) &
            (All_spot_data_mutate["replicate"] == rep) &
            (All_spot_data_mutate["FOV"] == fov) 
        ]

        t = group["Time"].values
        um2_biomass = group["um2_biomass"].values
        um2_biomass_fit = exp_growth(t, row["um2_biomass0"], row["k"])

        ax.plot(t, um2_biomass, 
                'o', label="Observed", color='blue')
        ax.plot(t, um2_biomass_fit,
                '-', label="Fitted", color='orange')
        ax.set_title(f"Specie={specie}, Condition={condition},\nR²={row['r_squared']:.3f}, N={rep}, FOV={fov}")
        ax.set_xlabel("Time")
        ax.set_ylabel("um2_biomass")
        ax.legend()
        
        set_mytheme_paper(ax)

    plt.tight_layout(rect=[0, 0, 1, 0.95])
    
    if output_folder:
        plt.savefig(os.path.join(output_folder, "fitted_biomass_transition.png"), 
                    dpi=600, transparent=True) #png
        plt.savefig(os.path.join(output_folder, "fitted_biomass_transition.svg"), 
                    dpi=600, transparent=True) # svg
        print(f"Saved to fitted_biomass_transition.png and .svg")
    
    if IF_show:
        plt.show()
    else:
        plt.close()


In [ ]:
output_folder="./result(plots)"
os.makedirs(output_folder, exist_ok=True)
plot_fitting_result(fit_df, All_spot_data_mutate_limited, output_folder=output_folder, IF_show=False)

# 5. save

In [ ]:
output_folder="./processed_data"
os.makedirs(output_folder, exist_ok=True)
fit_df.to_csv(os.path.join(output_folder, "1_fitted_allspots_params.csv"), index=False)